# Smart Waste Collection Simulation
# Simulation Model Development
#
# Objective:
# Build and validate a Monte Carlo simulation model that uses
# the empirical random variables prepared by Member 4 to simulate
# daily smart-bin fullness while keeping the collection decision
# rule configurable.

In [ ]:
# ---------------------------------------------------------
# Step 1: Load simulation inputs
# ---------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

rv1 = pd.read_csv(
    "simulation_RV1_accumulation_occurrence.csv"
)

rv2 = pd.read_csv(
    "simulation_RV2_increase_magnitude.csv"
)

print("RV1 loaded:", rv1.shape)
print("RV2 loaded:", rv2.shape)

In [ ]:
# ---------------------------------------------------------
# Step 2: Validate simulation inputs
# ---------------------------------------------------------

rv1_prob_cols = [
    "Increase",
    "No Change"
]

rv2_prob_cols = [
    "increase_2",
    "increase_4",
    "increase_6",
    "increase_8",
    "increase_10"
]

# Check missing values
print("RV1 missing values:")
print(rv1.isna().sum().sum())

print("\nRV2 missing values:")
print(rv2.isna().sum().sum())


# Check probability sums
rv1["probability_sum"] = (
    rv1[rv1_prob_cols].sum(axis=1)
)

rv2["probability_sum"] = (
    rv2[rv2_prob_cols].sum(axis=1)
)

print("\nRV1 probability sums:")
print(
    rv1[
        ["previous_fullness", "probability_sum"]
    ]
)

print("\nRV2 probability sums:")
print(
    rv2[
        ["previous_fullness", "probability_sum"]
    ]
)


# Check probability range
rv1_valid_range = (
    (rv1[rv1_prob_cols] >= 0) &
    (rv1[rv1_prob_cols] <= 1)
).all().all()

rv2_valid_range = (
    (rv2[rv2_prob_cols] >= 0) &
    (rv2[rv2_prob_cols] <= 1)
).all().all()

print("\nAll RV1 probabilities between 0 and 1:")
print(rv1_valid_range)

print("\nAll RV2 probabilities between 0 and 1:")
print(rv2_valid_range)

In [ ]:
# ---------------------------------------------------------
# Step 3: Create RV1 sampling function
# ---------------------------------------------------------

def sample_accumulation_event(current_fullness, rng):

    # Find the probability row for the current fullness
    row = rv1[
        rv1["previous_fullness"] == current_fullness
    ].iloc[0]

    # Randomly select Increase or No Change
    event = rng.choice(
        ["Increase", "No Change"],
        p=[
            row["Increase"],
            row["No Change"]
        ]
    )

    return event


# Create a reproducible random generator
rng = np.random.default_rng(42)


# Test RV1 from each possible fullness state
for state in [0, 2, 4, 6, 8, 10]:

    result = sample_accumulation_event(
        state,
        rng
    )

    print(
        f"Current fullness {state} -> {result}"
    )

In [ ]:
# ---------------------------------------------------------
# Step 4: Create RV2 sampling function
# ---------------------------------------------------------

def sample_increase_amount(current_fullness, rng):

    # Find RV2 row for current fullness
    row = rv2[
        rv2["previous_fullness"] == current_fullness
    ].iloc[0]

    # Possible increase amounts
    increase_amounts = [2, 4, 6, 8, 10]

    # Their probabilities
    probabilities = [
        row["increase_2"],
        row["increase_4"],
        row["increase_6"],
        row["increase_8"],
        row["increase_10"]
    ]

    # Randomly select increase amount
    increase = rng.choice(
        increase_amounts,
        p=probabilities
    )

    return int(increase)


# Use a fresh generator for this test
rng = np.random.default_rng(42)


# Test RV2 for every state where an increase is possible
for state in [0, 2, 4, 6, 8]:

    result = sample_increase_amount(
        state,
        rng
    )

    print(
        f"Current fullness {state} -> Increase by {result}"
    )

In [ ]:
# ---------------------------------------------------------
# Step 5: Simulate one day of accumulation
# ---------------------------------------------------------

def simulate_accumulation_day(current_fullness, rng):

    # Step 1: Sample RV1
    event = sample_accumulation_event(
        current_fullness,
        rng
    )

    # Step 2: If there is no increase
    if event == "No Change":

        increase_amount = 0
        new_fullness = current_fullness

    # Step 3: If fullness increases
    else:

        increase_amount = sample_increase_amount(
            current_fullness,
            rng
        )

        new_fullness = (
            current_fullness + increase_amount
        )

    return {
        "start_fullness": current_fullness,
        "accumulation_event": event,
        "increase_amount": increase_amount,
        "new_fullness": new_fullness
    }


# Fresh random generator
rng = np.random.default_rng(42)


# Test one simulated day from every possible state
for state in [0, 2, 4, 6, 8, 10]:

    result = simulate_accumulation_day(
        state,
        rng
    )

    print(result)

In [ ]:
# ---------------------------------------------------------
# Step 6: Apply configurable collection decision rule
# ---------------------------------------------------------

def apply_collection_rule(
    pre_collection_fullness,
    collection_threshold
):

    if pre_collection_fullness >= collection_threshold:

        collected = True
        end_fullness = 0

    else:

        collected = False
        end_fullness = pre_collection_fullness

    return {
        "collected": collected,
        "end_fullness": end_fullness
    }


# Test threshold
collection_threshold = 8

# Test every possible fullness state
for state in [0, 2, 4, 6, 8, 10]:

    result = apply_collection_rule(
        state,
        collection_threshold
    )

    print(
        f"Fullness {state} -> "
        f"Collected: {result['collected']}, "
        f"End fullness: {result['end_fullness']}"
    )

In [ ]:
# ---------------------------------------------------------
# Step 7: Complete one-day simulation
# ---------------------------------------------------------

def simulate_one_day(
    current_fullness,
    collection_threshold,
    rng
):

    # 1. Simulate natural accumulation
    accumulation = simulate_accumulation_day(
        current_fullness,
        rng
    )

    pre_collection_fullness = (
        accumulation["new_fullness"]
    )

    # 2. Apply collection decision
    collection = apply_collection_rule(
        pre_collection_fullness,
        collection_threshold
    )

    # 3. Return complete daily record
    return {
        "start_fullness":
            current_fullness,

        "accumulation_event":
            str(accumulation["accumulation_event"]),

        "increase_amount":
            accumulation["increase_amount"],

        "pre_collection_fullness":
            pre_collection_fullness,

        "collected":
            collection["collected"],

        "end_fullness":
            collection["end_fullness"]
    }


# Fresh generator
rng = np.random.default_rng(42)

collection_threshold = 8


# Test from every possible starting state
for state in [0, 2, 4, 6, 8, 10]:

    result = simulate_one_day(
        state,
        collection_threshold,
        rng
    )

    print(result)

In [ ]:
# ---------------------------------------------------------
# Step 8: Simulate one bin over multiple days
# ---------------------------------------------------------

def simulate_bin(
    number_of_days,
    starting_fullness,
    collection_threshold,
    seed
):

    # Create reproducible random generator
    rng = np.random.default_rng(seed)

    # Set initial fullness
    current_fullness = starting_fullness

    # Store daily simulation results
    records = []

    # Simulate each day
    for day in range(1, number_of_days + 1):

        result = simulate_one_day(
            current_fullness,
            collection_threshold,
            rng
        )

        # Add day number
        result["day"] = day

        # Save result
        records.append(result)

        # Today's ending fullness becomes
        # tomorrow's starting fullness
        current_fullness = result["end_fullness"]

    # Convert results to DataFrame
    return pd.DataFrame(records)


# Run a small 30-day test
test_simulation = simulate_bin(
    number_of_days=30,
    starting_fullness=0,
    collection_threshold=8,
    seed=42
)

print(test_simulation.to_string(index=False))

In [ ]:
# ---------------------------------------------------------
# Step 9: Validate multi-day simulation logic
# ---------------------------------------------------------

valid_states = [0, 2, 4, 6, 8, 10]

# 1. Check all fullness values are valid states
valid_start_states = (
    test_simulation["start_fullness"]
    .isin(valid_states)
    .all()
)

valid_pre_collection_states = (
    test_simulation["pre_collection_fullness"]
    .isin(valid_states)
    .all()
)

valid_end_states = (
    test_simulation["end_fullness"]
    .isin(valid_states)
    .all()
)


# 2. Check collected bins always end at 0
collection_reset_valid = (
    test_simulation.loc[
        test_simulation["collected"],
        "end_fullness"
    ] == 0
).all()


# 3. Check each day's end fullness becomes
#    the next day's start fullness
continuity_valid = (
    test_simulation["end_fullness"].iloc[:-1].to_numpy()
    ==
    test_simulation["start_fullness"].iloc[1:].to_numpy()
).all()


# 4. Check fullness never exceeds maximum state 10
maximum_valid = (
    test_simulation["pre_collection_fullness"].max()
    <= 10
)


print("Valid start states:", valid_start_states)

print(
    "Valid pre-collection states:",
    valid_pre_collection_states
)

print("Valid end states:", valid_end_states)

print(
    "Collected bins reset to 0:",
    collection_reset_valid
)

print(
    "Daily continuity valid:",
    continuity_valid
)

print(
    "No fullness above 10:",
    maximum_valid
)

In [ ]:
# ---------------------------------------------------------
# Step 10: Run multiple Monte Carlo simulations
# ---------------------------------------------------------

def run_monte_carlo(
    number_of_runs,
    number_of_days,
    starting_fullness,
    collection_threshold,
    base_seed
):

    all_results = []

    for run in range(1, number_of_runs + 1):

        # Give each run a different but reproducible seed
        run_seed = base_seed + run

        simulation = simulate_bin(
            number_of_days=number_of_days,
            starting_fullness=starting_fullness,
            collection_threshold=collection_threshold,
            seed=run_seed
        )

        # Add simulation run number
        simulation["run"] = run

        all_results.append(simulation)

    # Combine all runs into one DataFrame
    monte_carlo_results = pd.concat(
        all_results,
        ignore_index=True
    )

    return monte_carlo_results


# Small Monte Carlo test
mc_test = run_monte_carlo(
    number_of_runs=10,
    number_of_days=30,
    starting_fullness=0,
    collection_threshold=8,
    base_seed=42
)


print("Simulation shape:")
print(mc_test.shape)

print("\nNumber of runs:")
print(mc_test["run"].nunique())

print("\nDays per run:")
print(mc_test.groupby("run")["day"].count())

print("\nFirst 10 rows:")
print(mc_test.head(10).to_string(index=False))

In [ ]:
# ---------------------------------------------------------
# Step 11: Calculate KPIs for each Monte Carlo run
# ---------------------------------------------------------

run_summary = (
    mc_test
    .groupby("run")
    .agg(
        collections=(
            "collected",
            "sum"
        ),

        days_at_state_10=(
            "pre_collection_fullness",
            lambda x: (x == 10).sum()
        ),

        average_fullness=(
            "pre_collection_fullness",
            "mean"
        )
    )
    .reset_index()
)


# Calculate average fullness specifically when collected
collection_fullness = (
    mc_test[
        mc_test["collected"]
    ]
    .groupby("run")[
        "pre_collection_fullness"
    ]
    .mean()
    .rename("average_collection_fullness")
    .reset_index()
)


# Merge with run summary
run_summary = run_summary.merge(
    collection_fullness,
    on="run",
    how="left"
)


print(run_summary.to_string(index=False))

In [ ]:
# ---------------------------------------------------------
# Step 12: Run full base Monte Carlo simulation
# ---------------------------------------------------------

base_results = run_monte_carlo(
    number_of_runs=1000,
    number_of_days=365,
    starting_fullness=0,
    collection_threshold=8,
    base_seed=42
)

print("Simulation shape:")
print(base_results.shape)

print("\nNumber of runs:")
print(base_results["run"].nunique())

print("\nDays per run:")
print(
    base_results.groupby("run")["day"]
    .count()
    .describe()
)

print("\nTotal simulated bin-days:")
print(len(base_results))

In [ ]:
# ---------------------------------------------------------
# Step 13: Calculate KPIs for full base simulation
# ---------------------------------------------------------

base_run_summary = (
    base_results
    .groupby("run")
    .agg(
        collections=(
            "collected",
            "sum"
        ),

        days_at_state_10=(
            "pre_collection_fullness",
            lambda x: (x == 10).sum()
        ),

        average_fullness=(
            "pre_collection_fullness",
            "mean"
        )
    )
    .reset_index()
)


# Average fullness when collection occurs
base_collection_fullness = (
    base_results[
        base_results["collected"]
    ]
    .groupby("run")[
        "pre_collection_fullness"
    ]
    .mean()
    .rename("average_collection_fullness")
    .reset_index()
)


# Merge results
base_run_summary = base_run_summary.merge(
    base_collection_fullness,
    on="run",
    how="left"
)


# Display summary across the 1,000 simulation runs
print(
    base_run_summary[
        [
            "collections",
            "days_at_state_10",
            "average_fullness",
            "average_collection_fullness"
        ]
    ].describe()
)

In [ ]:
# ---------------------------------------------------------
# Step 14: Validate full base simulation
# ---------------------------------------------------------

valid_states = [0, 2, 4, 6, 8, 10]

# Check valid states
valid_start_states = (
    base_results["start_fullness"]
    .isin(valid_states)
    .all()
)

valid_pre_collection_states = (
    base_results["pre_collection_fullness"]
    .isin(valid_states)
    .all()
)

valid_end_states = (
    base_results["end_fullness"]
    .isin(valid_states)
    .all()
)

# Check collection reset
collection_reset_valid = (
    base_results.loc[
        base_results["collected"],
        "end_fullness"
    ] == 0
).all()

# Check maximum fullness
maximum_valid = (
    base_results["pre_collection_fullness"].max()
    <= 10
)

# Check every run has exactly 365 days
days_per_run_valid = (
    base_results
    .groupby("run")["day"]
    .count()
    .eq(365)
    .all()
)

# Check continuity separately within each run
continuity_errors = 0

for run_id, group in base_results.groupby("run"):

    group = group.sort_values("day")

    previous_end = (
        group["end_fullness"]
        .iloc[:-1]
        .to_numpy()
    )

    next_start = (
        group["start_fullness"]
        .iloc[1:]
        .to_numpy()
    )

    continuity_errors += (
        previous_end != next_start
    ).sum()

continuity_valid = (
    continuity_errors == 0
)

print("Valid start states:", valid_start_states)

print(
    "Valid pre-collection states:",
    valid_pre_collection_states
)

print("Valid end states:", valid_end_states)

print(
    "Collected bins reset to 0:",
    collection_reset_valid
)

print(
    "No fullness above 10:",
    maximum_valid
)

print(
    "Every run has 365 days:",
    days_per_run_valid
)

print(
    "Daily continuity valid:",
    continuity_valid
)

print(
    "Continuity errors:",
    continuity_errors
)

In [ ]:
# ---------------------------------------------------------
# Step 15: Validate RV1 sampling probabilities
# ---------------------------------------------------------

validation_runs = 10000

rv1_validation_results = []

# Test each possible fullness state
for state in [0, 2, 4, 6, 8, 10]:

    # Separate reproducible generator for each state
    rng_test = np.random.default_rng(
        100 + state
    )

    sampled_events = []

    for _ in range(validation_runs):

        event = sample_accumulation_event(
            state,
            rng_test
        )

        sampled_events.append(
            str(event)
        )

    # Calculate simulated Increase probability
    simulated_increase = (
        np.mean(
            np.array(sampled_events)
            == "Increase"
        )
    )

    # Get expected probability from RV1
    expected_increase = (
        rv1.loc[
            rv1["previous_fullness"] == state,
            "Increase"
        ].iloc[0]
    )

    rv1_validation_results.append(
        {
            "fullness_state": state,
            "expected_increase":
                expected_increase,

            "simulated_increase":
                simulated_increase,

            "difference":
                simulated_increase
                - expected_increase
        }
    )


rv1_validation = pd.DataFrame(
    rv1_validation_results
)

print(
    rv1_validation.to_string(
        index=False
    )
)

In [ ]:
# ---------------------------------------------------------
# Step 16: Validate RV2 sampling probabilities
# ---------------------------------------------------------

validation_runs = 10000

rv2_validation_results = []

increase_amounts = [2, 4, 6, 8, 10]

for state in [0, 2, 4, 6, 8]:

    # Reproducible generator
    rng_test = np.random.default_rng(
        200 + state
    )

    sampled_amounts = []

    for _ in range(validation_runs):

        amount = sample_increase_amount(
            state,
            rng_test
        )

        sampled_amounts.append(amount)

    # Compare each possible increase amount
    for amount in increase_amounts:

        expected_probability = (
            rv2.loc[
                rv2["previous_fullness"] == state,
                f"increase_{amount}"
            ].iloc[0]
        )

        simulated_probability = (
            np.mean(
                np.array(sampled_amounts)
                == amount
            )
        )

        rv2_validation_results.append(
            {
                "fullness_state": state,
                "increase_amount": amount,
                "expected_probability":
                    expected_probability,
                "simulated_probability":
                    simulated_probability,
                "difference":
                    simulated_probability
                    - expected_probability
            }
        )


rv2_validation = pd.DataFrame(
    rv2_validation_results
)

print(
    rv2_validation.to_string(
        index=False
    )
)

In [ ]:
# ---------------------------------------------------------
# Step 17: Test simulation reproducibility
# ---------------------------------------------------------

base_results_repeat = run_monte_carlo(
    number_of_runs=1000,
    number_of_days=365,
    starting_fullness=0,
    collection_threshold=8,
    base_seed=42
)

# Compare original and repeated simulation
reproducible = base_results.equals(
    base_results_repeat
)

print(
    "Repeated simulation identical:",
    reproducible
)

print(
    "Original rows:",
    len(base_results)
)

print(
    "Repeated rows:",
    len(base_results_repeat)
)

In [ ]:
# ---------------------------------------------------------
# Step 18: Create base simulation summary
# ---------------------------------------------------------

base_summary = pd.DataFrame({
    "metric": [
        "Number of Monte Carlo runs",
        "Days per run",
        "Starting fullness",
        "Collection threshold",
        "Base random seed",
        "Mean collections per year",
        "Median collections per year",
        "Minimum collections",
        "Maximum collections",
        "Mean days at state 10",
        "Mean pre-collection fullness",
        "Mean fullness when collected"
    ],

    "value": [
        1000,
        365,
        0,
        8,
        42,
        base_run_summary["collections"].mean(),
        base_run_summary["collections"].median(),
        base_run_summary["collections"].min(),
        base_run_summary["collections"].max(),
        base_run_summary["days_at_state_10"].mean(),
        base_run_summary["average_fullness"].mean(),
        base_run_summary[
            "average_collection_fullness"
        ].mean()
    ]
})


print(base_summary.to_string(index=False))

In [ ]:
# ---------------------------------------------------------
# Step 19: Export Member 5 simulation outputs
# ---------------------------------------------------------

# Detailed daily simulation results
base_results.to_csv(
    "simulation_base_results.csv",
    index=False
)

# Results for each Monte Carlo run
base_run_summary.to_csv(
    "simulation_base_run_summary.csv",
    index=False
)

# Overall base-model summary
base_summary.to_csv(
    "simulation_base_summary.csv",
    index=False
)


print("Files exported successfully:")

print(
    "1. simulation_base_results.csv",
    base_results.shape
)

print(
    "2. simulation_base_run_summary.csv",
    base_run_summary.shape
)

print(
    "3. simulation_base_summary.csv",
    base_summary.shape
)